# 08 — One neuron: a weighted sum with a bend

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/omarahmedZa/ai-ml-course/blob/main/notebooks/008-the-neuron.ipynb)

A neuron is the smallest piece of a network: multiply each input by a weight, add a bias, then pass the result through a bend called ReLU. In this notebook you compute one by hand, plot its output, and drag the weight to watch the bend move. Then you break it.

## 1. The numbers

The neuron takes two numbers in and gives one number out. The weights say how much each input matters, the bias shifts the whole sum up or down, and ReLU clips anything negative to zero.

In [ ]:
import numpy as np                                    # the array library, preinstalled on Colab

x = np.array([1.0, 2.0])                              # one input: two features
w = np.array([0.5, -1.5])                             # one weight per feature
b = 0.25                                              # the bias: a constant added at the end

z = x @ w + b                                         # the raw score: dot product plus bias
a = max(0.0, z)                                       # ReLU: keep positives, replace negatives with 0
print("x =", x)                                       # show the input
print("z =", z)                                       # -2.25, the score before the bend
print("a = relu(z) =", a)                             # 0.0, the negative score is clipped

**Line by line**

- `import numpy as np` — loads NumPy, the array library Colab already has.
- `x = ...` — the input: two features.
- `w = ...` — the learned weights, one per feature.
- `b = 0.25` — a constant that shifts the sum after the dot product.
- `z = x @ w + b` — `@` is the dot product; the bias is added afterwards.
- `a = max(0.0, z)` — ReLU, written plainly: the larger of zero and the score.
- `print("x = ...")` — shows the input that was used.
- `print("z = ...")` — the raw score before the bend.
- `print("a = ...")` — the output after the bend.

## 2. Step by step

The `@` symbol hides the arithmetic, so here it is one feature at a time, wrapped in a function you can call repeatedly. Each call prints every term, the raw score, and the output after the bend.

In [ ]:
def neuron(x, w, b):                                  # a reusable neuron: input, weights, bias
    total = 0.0                                       # start the running sum at zero
    for i in range(len(x)):                           # one turn per feature
        term = x[i] * w[i]                            # multiply the feature by its weight
        total = total + term                          # fold the term into the sum
        print(f"  feature {i}: {x[i]:+.1f} x {w[i]:+.1f} = {term:+.2f}")   # show the term
    z = total + b                                     # add the bias to finish the raw score
    a = max(0.0, z)                                   # bend it with ReLU
    print(f"  z = {total:.2f} + {b:.2f} = {z:.2f} -> a = {a:.2f}")   # show the bend
    return z, a                                       # hand back both numbers

for x_i in [np.array([1., 2.]), np.array([-1., -2.])]:   # two different inputs
    print("input", x_i)                               # label the block
    neuron(x_i, w, b)                                 # run the same neuron twice

**Line by line**

- `def neuron(x, w, b)` — the same computation, packaged so it can be reused.
- `total = 0.0` — the accumulator for the dot product.
- `for i in range(len(x))` — one iteration per feature.
- `term = x[i] * w[i]` — feature times its own weight.
- `total = total + term` — add the term to the running sum.
- `print(f"...")` — an f-string; the braces are replaced by values.
- `z = total + b` — the bias finishes the raw score.
- `a = max(0.0, z)` — ReLU, applied to the single number.
- `print(f"  z = ...")` — prints the bias added and the bend applied.
- `return z, a` — both stages, so you can inspect either.
- `for x_i in [...]` — the same neuron on a positive and a negative input.
- `print("input", x_i)` — labels the block so the output stays readable.
- `neuron(x_i, w, b)` — the call itself, once per input.

## 3. Plot: output against input

Sweep one input from -4 to 4 and keep the weight and bias fixed. The raw score is a straight line; ReLU keeps the part above zero and flattens the rest, so the neuron is two straight pieces joined at a corner.

In [ ]:
import matplotlib.pyplot as plt                        # plotting library, preinstalled on Colab

w1 = w[0]                                             # keep one weight for a one-dimensional sweep
t = np.linspace(-4, 4, 201)                           # 201 input values from -4 to 4
z_line = w1 * t + b                                   # the raw score: a straight line
a_line = np.maximum(0.0, z_line)                      # ReLU flattens the negative half

plt.figure(figsize=(6, 3))                            # wide, short figure
plt.plot(t, z_line, color="tab:grey", linestyle="--", label="z = w*x + b")   # before the bend
plt.plot(t, a_line, color="tab:blue", linewidth=2, label="a = relu(z)")      # after the bend
plt.axhline(0, color="grey", linewidth=0.6)           # the zero line
plt.xlabel("input x")                                 # x-axis label
plt.ylabel("neuron output")                           # y-axis label
plt.title("ReLU: a straight line with its negative half flattened")   # the takeaway
plt.legend()                                          # name both curves
plt.show()                                            # render the figure

**Line by line**

- `import matplotlib.pyplot as plt` — Colab already ships the plotting library.
- `w1 = w[0]` — one weight, so the input can be swept on its own.
- `np.linspace(-4, 4, 201)` — 201 evenly spaced test inputs.
- `z_line = ...` — the raw score for every input at once.
- `np.maximum(0.0, z_line)` — ReLU applied to the whole array elementwise.
- `plt.figure(figsize=(6, 3))` — a wide, short canvas.
- `plt.plot(t, z_line, ...)` — the dashed grey line before the bend.
- `plt.plot(t, a_line, ...)` — the solid blue line after the bend.
- `plt.axhline(0, ...)` — marks where the clipping starts.
- `plt.xlabel(...)` — names the input axis.
- `plt.ylabel(...)` — names the output axis.
- `plt.title(...)` — states what the picture should teach.
- `plt.legend()` — matches each line to its label.
- `plt.show()` — draws the finished figure.

## 4. Interactive: turn the weight

Drag the slider and watch the line tilt. The dashed line is the raw score and the solid line is what the neuron actually outputs. The title reports the weight so you can match the shape to the number.

In [ ]:
from ipywidgets import interact                        # sliders that rerun a function

def show(w_value=0.5):                                # w_value comes from the slider
    z_i = w_value * t + b                             # raw score for every input at this weight
    a_i = np.maximum(0.0, z_i)                        # bend it with ReLU
    plt.figure(figsize=(6, 3))                        # fresh figure on every drag
    plt.plot(t, z_i, color="tab:grey", linestyle="--", label="z")   # the straight line
    plt.plot(t, a_i, color="tab:blue", linewidth=2, label="a = relu(z)")   # the output
    plt.axhline(0, color="grey", linewidth=0.6)       # the zero line
    plt.xlim(-4, 4); plt.ylim(-3, 5)                  # fixed axes, so nothing jumps
    plt.xlabel("input x")                              # x-axis label
    plt.ylabel("neuron output")                        # y-axis label
    plt.title(f"w = {w_value:.2f}: steeper weight, narrower live region")   # live readout
    plt.legend()                                       # name the curves
    plt.show()                                         # draw it

interact(show, w_value=(0.1, 3.0, 0.1))                # attach the slider

**Line by line**

- `from ipywidgets import interact` — Colab ships the widgets library.
- `def show(w_value=0.5)` — the function that redraws one frame.
- `z_i = ...` — recompute the raw score at the slider's weight.
- `a_i = ...` — ReLU applied to the new scores.
- `plt.figure(...)` — a new canvas, so frames do not stack.
- `plt.plot(t, z_i, ...)` — the dashed line before the bend.
- `plt.plot(t, a_i, ...)` — the solid line after the bend.
- `plt.axhline(0, ...)` — the zero line.
- `plt.xlim(...); plt.ylim(...)` — fixed axes keep the picture comparable.
- `plt.xlabel(...)` — names the input axis.
- `plt.ylabel(...)` — names the output axis.
- `plt.title(f"...")` — prints the current weight next to the shape.
- `plt.legend()` — matches each line to its label.
- `plt.show()` — draws the frame.
- `interact(show, w_value=(...))` — the slider, its range and its step.

## 5. When it breaks: the dead unit

ReLU only learns where its score is positive, because its slope is zero everywhere else. If every input arrives negative and the bias is not high enough, the score is never positive and the neuron outputs zero forever — a dead unit.

In [ ]:
w_dead = 2.0                                          # a positive weight ...
b_dead = -1.0                                         # ... and a bias that pulls the sum down
x_neg = np.linspace(-4, -0.5, 100)                    # every input is negative here
z_dead = w_dead * x_neg + b_dead                      # raw score: still below zero everywhere
a_dead = np.maximum(0.0, z_dead)                      # ReLU flattens all of it
slope = (z_dead > 0).astype(float)                    # ReLU's slope: 1 where z > 0, else 0
print("largest raw score:", z_dead.max())             # -2.0, never positive
print("non-zero outputs:", int((a_dead > 0).sum()), "of", len(a_dead))   # 0 of 100
print("non-zero slopes:", int(slope.sum()), "-> nothing to learn from")  # 0

plt.figure(figsize=(6, 3))                            # same figure shape as before
plt.plot(x_neg, z_dead, color="tab:grey", linestyle="--", label="z")   # the line, entirely below 0
plt.plot(x_neg, a_dead, color="tab:red", linewidth=2, label="a = relu(z)")   # flat at zero
plt.axhline(0, color="grey", linewidth=0.6)           # the zero line
plt.xlabel("input x (all negative)")                  # x-axis label
plt.ylabel("neuron output")                           # y-axis label
plt.title("all-negative inputs -> dead unit, output stuck at 0")   # the failure
plt.legend()                                          # name the curves
plt.show()                                            # render

**Line by line**

- `w_dead` — a positive weight.
- `b_dead` — a bias that pulls the score down below zero.
- `x_neg = ...` — 100 inputs, all of them negative.
- `z_dead = ...` — the raw score for each, all below zero.
- `np.maximum(0.0, z_dead)` — ReLU erases every value.
- `slope = (z_dead > 0)` — the derivative of ReLU: 1 above zero, 0 below.
- `print("largest raw score", ...)` — the best case is still negative.
- `print("non-zero outputs", ...)` — none of the hundred survive.
- `print("non-zero slopes", ...)` — zero slopes mean zero gradient, so no learning.
- `plt.figure(...)` — the same figure shape as before.
- `plt.plot(x_neg, z_dead, ...)` — the dashed line, entirely below the axis.
- `plt.plot(x_neg, a_dead, ...)` — the output lying flat on zero.
- `plt.axhline(0, ...)` — the zero line.
- `plt.xlabel(...)` — notes that every input is negative.
- `plt.ylabel(...)` — names the output axis.
- `plt.title(...)` — names the failure in one line.
- `plt.legend()` — matches each line to its label.
- `plt.show()` — draws the figure.

## Your turn

In section 4, drag the weight down to 0.1 and find the input value where the bend happens. Then change `b_dead` in section 5 to `+2.0` and predict how many non-zero outputs you will see before you rerun.